# Run Attack Strength Curve Probe Shard

本 notebook 只负责在 Colab 冷启动环境中运行“内部三方法多强度攻击曲线”的一个 shard, 并检查该 shard 结果是否完整。

本 notebook 不做 shard 聚合, 不重建 paper artifacts。聚合请使用 `paper_workflow/aggregate_attack_strength_curve_probe_shards.ipynb`。


## User Config

参数配置应先参考已落盘的历史 shard 诊断信息。若首次运行没有历史结果, 使用默认 A100-40G 激进配置进行 smoke, 再根据 runtime diagnostics 调整。

In [ ]:
# 可修改配置区
REPO_URL = "https://github.com/RICHAAARC/TSTW-VW.git"  # Colab 冷启动克隆仓库使用的 HTTPS 地址。
REPO_BRANCH = "explicit-sync"  # 需要验证的分支名称。
DRIVE_ROOT = "/content/drive/MyDrive"  # Google Drive 挂载后的根目录。
RESULT_ROOT = f"{DRIVE_ROOT}/TSTW/results"  # TSTW 受治理结果根目录。
LOCAL_ROOT = "/content/TSTW_runtime"  # Colab session-local 工作区。
REPO_DIR = f"{LOCAL_ROOT}/repo"  # 仓库克隆到 Colab 本地后的路径。

PROCESSED_DATASET_KEY = "real_video_vae_latent_probe_davis2017_trainval480p_256x256_32f_8fps_paper_low_fpr"  # Drive 侧 processed dataset key。
PROCESSED_DATASET_ROOT = f"{DRIVE_ROOT}/TSTW/datasets/processed/{PROCESSED_DATASET_KEY}"  # Drive 侧 processed dataset 根目录。
LOCAL_DATASET_ROOT = f"{LOCAL_ROOT}/datasets/{PROCESSED_DATASET_KEY}"  # session-local processed dataset 根目录。
LOCAL_DATASET_MANIFEST = f"{LOCAL_DATASET_ROOT}/dataset_manifest.json"  # runner 显式消费的 dataset manifest。
MODEL_ID = "stabilityai/sd-vae-ft-mse"  # session-only AutoencoderKL 来源, 不写入 Drive 作为正式模型仓库。
MODEL_REVISION = "main"  # Hugging Face 模型 revision。
LOCAL_MODEL_ROOT = f"{LOCAL_ROOT}/session_models/autoencoder_kl"  # session-local VAE 模型目录。

H264_CRFS = "18,23,28,33,38"  # H.264 横轴强度, CRF 越大压缩越强。
H265_CRFS = "20,25,30,35,40"  # H.265 横轴强度, CRF 越大压缩越强。
TEMPORAL_CROP_KEEP_RATIOS = "0.90,0.75,0.60,0.50"  # temporal_crop 横轴, keep_ratio 越小攻击越强。
FRAME_DROP_RATES = "0.10,0.25,0.40,0.50"  # frame_dropping 横轴, drop_rate 越大攻击越强。
TARGET_FRAME_COUNT = 32  # processed video clip 帧数, 用于把 keep_ratio 转换为 crop_length。

# A100-40G 激进初始配置。首次建议 SAMPLES_PER_ROLE=8, SHARD_COUNT=1 做 smoke。
SAMPLES_PER_ROLE = 8  # smoke 配置: 小样本验证链路; 全量论文级运行再改为 100。
SHARD_COUNT = 1  # smoke 配置: 单 shard 验证; 全量运行再改为 4 或更多。
SHARD_INDEX = 0  # smoke 配置: 单 shard 时固定为 0。
BATCH_SIZE_FRAMES = 64  # smoke 配置: 保守 frame batch; A100-40G 全量可尝试 128。
WORKER_COUNT = 1  # cross-event VAE batching 当前要求 worker_count == 1。
VIDEO_IO_WORKER_COUNT = 4  # smoke 配置: 降低并发便于定位错误; 全量可尝试 8。
ATTACK_WORKER_COUNT = 4  # smoke 配置: 降低并发便于定位错误; 全量可尝试 8。
CROSS_EVENT_VAE_BATCHING_ENABLED = True  # 是否启用跨事件 VAE batching。
CROSS_EVENT_VAE_DECODE_BATCH_SIZE = 4  # smoke 配置: 保守 decode request batch; A100-40G 全量可尝试 12。
CROSS_EVENT_VAE_ENCODE_BATCH_SIZE = 4  # smoke 配置: 保守 encode request batch; A100-40G 全量可尝试 12。

RUN_REPOSITORY_SMOKE_TESTS = True  # 是否先运行攻击强度曲线相关轻量测试。
RUN_PREPARE_DATASET = True  # 是否复制 processed dataset 到 session-local。
RUN_PREPARE_MODEL = True  # 是否准备 session-only VAE 模型。
RUN_INTERNAL_MULTI_STRENGTH_SWEEP = True  # 是否运行当前 shard。
OVERWRITE_EXISTING_OUTPUT = True  # 当前 shard 输出目录存在时是否覆盖。


## Mount Google Drive

In [ ]:
from google.colab import drive

drive.mount('/content/drive')


## Prepare Local Workspace And Repository

In [ ]:
import json
import shutil
import subprocess
import sys
from pathlib import Path


def run_command(command, cwd=None, check=True):
    """运行命令并实时打印 stdout/stderr。"""
    print("$", " ".join(map(str, command)), flush=True)
    completed = subprocess.run(command, cwd=cwd, text=True, capture_output=True)
    if completed.stdout:
        print(completed.stdout, flush=True)
    if completed.stderr:
        print(completed.stderr, file=sys.stderr, flush=True)
    if check and completed.returncode != 0:
        raise RuntimeError("命令失败: " + " ".join(map(str, command)))
    return completed

local_root = Path(LOCAL_ROOT)
if local_root.exists():
    shutil.rmtree(local_root)
local_root.mkdir(parents=True, exist_ok=True)

run_command(["git", "clone", "--branch", REPO_BRANCH, "--depth", "1", REPO_URL, REPO_DIR])
short_commit = run_command(["git", "rev-parse", "--short", "HEAD"], cwd=REPO_DIR).stdout.strip()
print(json.dumps({"repo_dir": REPO_DIR, "short_commit": short_commit}, ensure_ascii=False, indent=2))


## Inspect Existing Shard Diagnostics

该步骤读取历史落盘结果, 帮助判断当前参数是否应调低或调高。首次运行没有历史结果时会给出空结果。

In [ ]:
def inspect_existing_attack_strength_runs(result_root):
    """读取已有 shard manifest 和轻量 runtime diagnostics。"""
    shard_root = Path(result_root) / "attack_strength_curve_probe" / "shard_runs"
    summaries = []
    if not shard_root.exists():
        return summaries
    for family_root in sorted([p for p in shard_root.iterdir() if p.is_dir()], key=lambda p: p.stat().st_mtime, reverse=True)[:12]:
        manifest_candidates = list((family_root / "artifacts").glob("attack_strength_*manifest.json"))
        summary = {"family_root": str(family_root), "name": family_root.name}
        if manifest_candidates:
            try:
                manifest = json.loads(manifest_candidates[0].read_text(encoding="utf-8"))
                summary.update({
                    "record_count": manifest.get("record_count"),
                    "source_mode": manifest.get("source_mode"),
                    "shard_count": manifest.get("shard_count"),
                    "shard_index": manifest.get("shard_index"),
                    "samples_per_role": manifest.get("samples_per_role"),
                    "strength_points": len(manifest.get("strength_points", [])),
                })
            except Exception as exc:
                summary["manifest_error"] = repr(exc)
        timing_path = family_root / "runner_diagnostics" / "runtime_profile" / "run_timing_summary.json"
        if timing_path.exists():
            try:
                summary["run_timing_summary"] = json.loads(timing_path.read_text(encoding="utf-8"))
            except Exception as exc:
                summary["timing_error"] = repr(exc)
        summaries.append(summary)
    return summaries

existing_run_summaries = inspect_existing_attack_strength_runs(RESULT_ROOT)
print(json.dumps(existing_run_summaries, ensure_ascii=False, indent=2)[:12000])


## Install Dependencies

In [ ]:
run_command([
    sys.executable, "-m", "pip", "install", "-q",
    "matplotlib",
    "imageio[ffmpeg]",
    "diffusers",
    "transformers",
    "accelerate",
    "huggingface_hub",
    "safetensors",
], cwd=REPO_DIR)


## Prepare Processed Dataset

In [ ]:
if RUN_PREPARE_DATASET:
    drive_dataset_root = Path(PROCESSED_DATASET_ROOT)
    local_dataset_root = Path(LOCAL_DATASET_ROOT)
    if not drive_dataset_root.exists():
        raise FileNotFoundError(drive_dataset_root)
    if local_dataset_root.exists():
        shutil.rmtree(local_dataset_root)
    print(json.dumps({"copy_dataset_from": str(drive_dataset_root), "copy_dataset_to": str(local_dataset_root)}, ensure_ascii=False, indent=2))
    shutil.copytree(drive_dataset_root, local_dataset_root)
    if not Path(LOCAL_DATASET_MANIFEST).exists():
        raise FileNotFoundError(LOCAL_DATASET_MANIFEST)
else:
    print("已跳过 processed dataset 复制。")


## Prepare Session VAE Model

In [ ]:
if RUN_PREPARE_MODEL:
    run_command([
        sys.executable,
        "scripts/prepare_models/prepare_session_autoencoder_kl.py",
        "--model-id", MODEL_ID,
        "--revision", MODEL_REVISION,
        "--local-model-root", LOCAL_MODEL_ROOT,
        "--session-manifest-path", f"{LOCAL_ROOT}/session_model_manifest.json",
    ], cwd=REPO_DIR)
else:
    print("已跳过 session VAE 模型准备。")


## Verify Repository Contract

In [ ]:
if RUN_REPOSITORY_SMOKE_TESTS:
    run_command([
        sys.executable,
        "-m",
        "pytest",
        "-q",
        "tests/constraints/test_attack_strength_curve_probe_builder.py",
    ], cwd=REPO_DIR)
else:
    print("已跳过仓库轻量约束测试。")


## Run Current Shard

In [ ]:
internal_sweep_output = None
if RUN_INTERNAL_MULTI_STRENGTH_SWEEP:
    local_run_root = f"{LOCAL_ROOT}/runs/attack_strength_curve_probe_internal_sweep_sc{SHARD_COUNT:02d}_si{SHARD_INDEX:02d}"
    sweep_args = [
        "scripts/package_results/run_internal_attack_strength_sweep.py",
        "--result-root", RESULT_ROOT,
        "--run-root", local_run_root,
        "--dataset-manifest", LOCAL_DATASET_MANIFEST,
        "--local-dataset-root", LOCAL_DATASET_ROOT,
        "--vae-model-local-path", LOCAL_MODEL_ROOT,
        "--short-commit", short_commit,
        "--samples-per-role", str(SAMPLES_PER_ROLE),
        "--shard-count", str(SHARD_COUNT),
        "--shard-index", str(SHARD_INDEX),
        "--batch-size-frames", str(BATCH_SIZE_FRAMES),
        "--worker-count", str(WORKER_COUNT),
        "--video-io-worker-count", str(VIDEO_IO_WORKER_COUNT),
        "--attack-worker-count", str(ATTACK_WORKER_COUNT),
        "--cross-event-vae-decode-batch-size", str(CROSS_EVENT_VAE_DECODE_BATCH_SIZE),
        "--cross-event-vae-encode-batch-size", str(CROSS_EVENT_VAE_ENCODE_BATCH_SIZE),
        "--target-frame-count", str(TARGET_FRAME_COUNT),
        "--h264-crfs", H264_CRFS,
        "--h265-crfs", H265_CRFS,
        "--temporal-crop-keep-ratios", TEMPORAL_CROP_KEEP_RATIOS,
        "--frame-drop-rates", FRAME_DROP_RATES,
    ]
    if CROSS_EVENT_VAE_BATCHING_ENABLED:
        sweep_args.append("--cross-event-vae-batching-enabled")
    if OVERWRITE_EXISTING_OUTPUT:
        sweep_args.append("--overwrite")
    completed = run_command([sys.executable, *sweep_args], cwd=REPO_DIR)
    internal_sweep_output = json.loads(completed.stdout)
else:
    print("已跳过内部多强度 sweep shard。")


## Check Current Shard Output

本 cell 只检查当前 shard 是否成功落盘, 不做聚合。

In [ ]:
if internal_sweep_output is None:
    raise RuntimeError("internal_sweep_output 为空, 当前 notebook 没有产生 shard 结果。")
shard_root = Path(internal_sweep_output["output_root"])
required_paths = {
    "record_path": shard_root / "records" / "attack_strength_event_scores.jsonl",
    "manifest_path": shard_root / "artifacts" / "attack_strength_internal_sweep_manifest.json",
}
manifest = json.loads(required_paths["manifest_path"].read_text(encoding="utf-8")) if required_paths["manifest_path"].exists() else {}
record_line_count = sum(1 for _ in required_paths["record_path"].open(encoding="utf-8")) if required_paths["record_path"].exists() else 0
completion_summary = {
    "result_flow": "attack_strength_shard_run",
    "output_root": str(shard_root),
    "required_paths": {key: path.exists() for key, path in required_paths.items()},
    "record_line_count": record_line_count,
    "manifest_record_count": manifest.get("record_count"),
    "source_mode": manifest.get("source_mode"),
    "shard_count": manifest.get("shard_count"),
    "shard_index": manifest.get("shard_index"),
    "short_commit": short_commit,
    "curve_attack_names": manifest.get("curve_attack_names"),
    "calibration_only_attack_names": manifest.get("calibration_only_attack_names"),
}
completion_summary["status"] = all(completion_summary["required_paths"].values()) and record_line_count > 0 and manifest.get("source_mode") == "full_multi_strength_sweep"
print(json.dumps(completion_summary, ensure_ascii=False, indent=2))
if not completion_summary["status"]:
    raise RuntimeError(completion_summary)
